In [1]:
import os
import pandas as pd
from pathlib import Path
import gc
import torch

# Run garbage collection
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Support Kaggle and local environments
kaggle_test_path = '/kaggle/input/competitions/name-that-llm/test.csv'
local_test_path = r'C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\test.csv'

if os.path.exists(kaggle_test_path):
    data_path = kaggle_test_path
elif os.path.exists('data/test.csv'):
    data_path = 'data/test.csv'
elif os.path.exists(local_test_path):
    data_path = local_test_path
else:
    data_path = 'test.csv'

print(f'Loading test data from: {data_path}')
test_df = pd.read_csv(data_path)
print(f'Test samples: {len(test_df)}')
test_df.head()

Loading test data from: data/test.csv
Test samples: 3


,id,text
0,00353fb1c541,"The more efficiently we use a resource, the mo..."
1,005f5a9a6139,"The Stern-Gerlach experiment, performed in 192..."
2,0062a53d5487,"Hundreds of millions of years ago, fluctuating..."


In [2]:
import os
import pickle

kaggle_le_path = '/kaggle/input/deberta-v3-finetuned/le.pkl'
local_le_path = r'C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\deberta_v3_finetuned\le.pkl'

if os.path.exists(kaggle_le_path):
    path = kaggle_le_path
elif os.path.exists('deberta_v3_finetuned/le.pkl'):
    path = 'deberta_v3_finetuned/le.pkl'
elif os.path.exists(local_le_path):
    path = local_le_path
else:
    path = 'le.pkl'

print(f'Loading label encoder from: {path}')
with open(path, 'rb') as f:
    le = pickle.load(f)

Loading label encoder from: deberta_v3_finetuned/le.pkl


In [3]:
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

kaggle_model_dir = '/kaggle/input/deberta-v3-finetuned'
local_model_dir = r'C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\new_data_fineTuned_model'

if os.path.exists(kaggle_model_dir) and os.path.exists(os.path.join(kaggle_model_dir, 'config.json')):
    LOCAL_DIR = kaggle_model_dir
elif os.path.exists('deberta_v3_finetuned'):
    LOCAL_DIR = 'deberta_v3_finetuned'
elif os.path.exists(local_model_dir):
    LOCAL_DIR = local_model_dir
else:
    LOCAL_DIR = 'deberta_v3_finetuned'

print(f'Loading model from: {LOCAL_DIR}')
tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

# Use torch_dtype=torch.float16 on GPU to cut VRAM usage in half
model_kwargs = {}
if torch.cuda.is_available():
    model_kwargs['torch_dtype'] = torch.float16

model = AutoModelForSequenceClassification.from_pretrained(LOCAL_DIR, **model_kwargs)
model.to(device)
model.eval()
print('Model loaded successfully.')

c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Loading model from: deberta_v3_finetuned


c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 106/106 [00:00<00:00, 3783.86it/s]

Model loaded successfully.


In [4]:
# Memory-safe batch inference function to prevent OOM errors on large test sets
from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return str(self.texts[idx])

dataset = TextDataset(test_df['text'].tolist())


In [5]:
import gc
import torch
from tqdm.auto import tqdm

BATCH_SIZE = 32  # Small batch size to strictly prevent OOM (both RAM and VRAM)
MAX_LENGTH = 256

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

all_preds = []

print(f'Running inference on {len(dataset)} items with batch_size={BATCH_SIZE}...')
with torch.no_grad():
    for batch_texts in tqdm(dataloader, desc='Predicting'):
        encoded = tokenizer(
            list(batch_texts),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )
        
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        batch_preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        all_preds.extend(batch_preds)
        
        # Clean up batch tensors immediately
        del encoded, input_ids, attention_mask, outputs, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Final cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'Total predictions generated: {len(all_preds)}')

Running inference on 3 items with batch_size=32...


Predicting: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]

Total predictions generated: 3


In [6]:
predictions_f = le.inverse_transform(all_preds)

submission = pd.DataFrame({
    'id': test_df['id'],
    'model': predictions_f
})

# Save output
submission.to_parquet('submission.parquet', index=False)
print(f'Saved submission.parquet with {len(submission)} rows.')
submission.head(10)

Saved submission.parquet with 3 rows.


,id,model
0,00353fb1c541,claude-opus-4-8
1,005f5a9a6139,gpt-5.6-luna
2,0062a53d5487,gemini-3.1-pro-preview


In [ ]:
# from psycopg.generators import fetch
# import re
# import numpy as np
# import pandas as pd
# import nltk
# nltk.download("punkt")
# from nltk.tokenize import sent_tokenize, word_tokenize

# features = pd.DataFrame(index=df.index)
# features["model"] = df["model"]
# text_col = "text"

In [ ]:
# # Character, word, and sentence counts
# features["char_count"] = df[text_col].apply(len)
# # Words & Sentences
# def get_words_and_sents(text):
#         sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
#         words = word_tokenize(text) if len(text.strip()) > 0 else [""]
#         return len(words), len(sents), sents, words

# tokenized_info = df[text_col].apply(get_words_and_sents)
# features["word_count"] = tokenized_info.apply(lambda x: x[0])
# features["sent_count"] = tokenized_info.apply(lambda x: x[1])
# # Sentence Length Mean & Variance
# def sent_stats(info):
#     _, num_sents, sents, _ = info
#     if num_sents == 0:
#         return 0.0, 0.0
#     lengths = [len(s.split()) for s in sents]
#     return float(np.mean(lengths)), float(np.var(lengths))

# sent_metrics = tokenized_info.apply(sent_stats)
# features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
# features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])
# # Avg Word Length
# features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)
# # ----------------- 2. Markdown Stylometry -----------------
# features["md_asterisk_count"] = df[text_col].apply(lambda x: x.count("*"))  # Bold or italics (* or **)
# # ----------------- 3. Punctuation Fingerprint -----------------
# features["punct_colon"] = df[text_col].apply(lambda x: x.count(":"))
# features["punct_semicolon"] = df[text_col].apply(lambda x: x.count(";"))
# features["punct_em_dash"] = df[text_col].apply(lambda x: x.count("—") + x.count("--"))
# features["punct_exclamation"] = df[text_col].apply(lambda x: x.count("!"))
# features["punct_question"] = df[text_col].apply(lambda x: x.count("?"))
# features["punct_comma"] = df[text_col].apply(lambda x: x.count(","))
# features["punct_period"] = df[text_col].apply(lambda x: x.count("."))
# features["punct_quotes"] = df[text_col].apply(lambda x: x.count('"') + x.count("'"))
# features["punct_parens"] = df[text_col].apply(lambda x: x.count("(") + x.count(")"))
# features["punct_brackets"] = df[text_col].apply(lambda x: x.count("[") + x.count("]"))
# # Punctuation Density (normalized by word count)
# total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
#                 features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
# features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)
# # ----------------- 4. Capitalization & Casing -----------------
# features["upper_char_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
# features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
# features["all_caps_words"] = df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
# features["camel_case_count"] = df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))
# # ----------------- 5. Lexical Diversity (Type-Token Ratio) -----------------
# def type_token_ratio(info):
#     num_words, _, _, words = info
#     if num_words == 0:
#         return 0.0
#     unique_words = set(w.lower() for w in words if w.isalnum())
#     return len(unique_words) / (num_words + 1e-5)

# features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)
# # Digits / Numbers presence
# features["digit_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
# features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)
# features.head()

In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder, StandardScaler

# #  Train-Test Split (stratified)
# df_train, df_test = train_test_split(features, test_size=0.2, random_state=42, stratify=features["model"])
# y_train = df_train["model"].values
# y_test = df_test["model"].values
# X_train_pre = df_train.iloc[:, 1:]
# X_test_pre = df_test.iloc[:, 1:]

# X_train_pre.shape, X_test_pre.shape
# # X_train.head()
# encoder = StandardScaler()
# X_train = encoder.fit_transform(X_train_pre)
# X_test = encoder.transform(X_test_pre)
# X_train.shape, X_test.shape
# X_train

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
# clf = LogisticRegression(max_iter=1000, C=2.0, verbose=1)

# clf.fit(X_train, y_train)
# y_pred = clf.predict(X_test)

# print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
# print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")
# # print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")



In [ ]:
# test = pd.read_csv(r"/kaggle/input/competitions/name-that-llm/test.csv")
# test.head()

# features = pd.DataFrame(index=df.index)
# # Character, word, and sentence counts
# features["char_count"] = test[text_col].apply(len)
# # Words & Sentences
# def get_words_and_sents(text):
#         sents = sent_tokenize(text) if len(text.strip()) > 0 else [""]
#         words = word_tokenize(text) if len(text.strip()) > 0 else [""]
#         return len(words), len(sents), sents, words

# tokenized_info = test[text_col].apply(get_words_and_sents)
# features["word_count"] = tokenized_info.apply(lambda x: x[0])
# features["sent_count"] = tokenized_info.apply(lambda x: x[1])
# # Sentence Length Mean & Variance
# def sent_stats(info):
#     _, num_sents, sents, _ = info
#     if num_sents == 0:
#         return 0.0, 0.0
#     lengths = [len(s.split()) for s in sents]
#     return float(np.mean(lengths)), float(np.var(lengths))

# sent_metrics = tokenized_info.apply(sent_stats)
# features["avg_sent_len"] = sent_metrics.apply(lambda x: x[0])
# features["var_sent_len"] = sent_metrics.apply(lambda x: x[1])
# # Avg Word Length
# features["avg_word_len"] = features["char_count"] / (features["word_count"] + 1e-5)
# # ----------------- 2. Markdown Stylometry -----------------
# features["md_asterisk_count"] = df[text_col].apply(lambda x: x.count("*"))  # Bold or italics (* or **)
# # ----------------- 3. Punctuation Fingerprint -----------------
# features["punct_colon"] = df[text_col].apply(lambda x: x.count(":"))
# features["punct_semicolon"] = df[text_col].apply(lambda x: x.count(";"))
# features["punct_em_dash"] = df[text_col].apply(lambda x: x.count("—") + x.count("--"))
# features["punct_exclamation"] = df[text_col].apply(lambda x: x.count("!"))
# features["punct_question"] = df[text_col].apply(lambda x: x.count("?"))
# features["punct_comma"] = df[text_col].apply(lambda x: x.count(","))
# features["punct_period"] = df[text_col].apply(lambda x: x.count("."))
# features["punct_quotes"] = df[text_col].apply(lambda x: x.count('"') + x.count("'"))
# features["punct_parens"] = df[text_col].apply(lambda x: x.count("(") + x.count(")"))
# features["punct_brackets"] = df[text_col].apply(lambda x: x.count("[") + x.count("]"))
# # Punctuation Density (normalized by word count)
# total_puncts = (features["punct_colon"] + features["punct_semicolon"] + 
#                 features["punct_comma"] + features["punct_period"] + features["punct_exclamation"])
# features["punct_density"] = total_puncts / (features["word_count"] + 1e-5)
# # ----------------- 4. Capitalization & Casing -----------------
# features["upper_char_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
# features["upper_char_ratio"] = features["upper_char_count"] / (features["char_count"] + 1e-5)
# features["all_caps_words"] = df[text_col].apply(lambda x: len(re.findall(r"\b[A-Z]{2,}\b", x)))
# features["camel_case_count"] = df[text_col].apply(lambda x: len(re.findall(r"\b[a-z]+[A-Z][a-zA-Z]*\b", x)))
# # ----------------- 5. Lexical Diversity (Type-Token Ratio) -----------------
# def type_token_ratio(info):
#     num_words, _, _, words = info
#     if num_words == 0:
#         return 0.0
#     unique_words = set(w.lower() for w in words if w.isalnum())
#     return len(unique_words) / (num_words + 1e-5)

# features["lexical_diversity_ttr"] = tokenized_info.apply(type_token_ratio)
# # Digits / Numbers presence
# features["digit_count"] = df[text_col].apply(lambda x: sum(1 for c in x if c.isdigit()))
# features["digit_ratio"] = features["digit_count"] / (features["char_count"] + 1e-5)
# features.shape

In [ ]:
# X_test_t = features.iloc[:, :].copy()
# X_test = encoder.transform(X_test_t)
# X_test_scaled_df = pd.DataFrame(X_test, columns=X_test_t.columns, index=X_test_t.index)
# X_test_scaled_df

In [ ]:

# predictions = clf.predict(X_test)
# print(predictions)
# print(le.inverse_transform(predictions)) 
# predictions_f = le.inverse_transform(predictions)
# # Submission file 
# submission = pd.DataFrame({
#     "id": test["id"],
#     "model": predictions_f
# })

# submission.to_parquet("submission.parquet", index=False)
# print(f"Saved submission.parquet with {len(submission)} rows.")
# submission.head(10)

In [ ]:
# import torch
# import numpy as np

# from tqdm.auto import tqdm
# from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

# # ==========================================
# # 1. DeBERTa Predictions
# # ==========================================

# model.eval()

# deberta_probs = []
# y_true = []

# with torch.no_grad():
#     progress_bar = tqdm(test_loader, desc="DeBERTa Predictions")

#     for batch in progress_bar:

#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"]

#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         )

#         # Convert logits into probabilities
#         probs = torch.softmax(outputs.logits, dim=1)

#         deberta_probs.extend(probs.cpu().numpy())
#         y_true.extend(labels.cpu().numpy())


# deberta_probs = np.array(deberta_probs)
# y_true = np.array(y_true)

# print("DeBERTa probability shape:", deberta_probs.shape)


# # ==========================================
# # 2. Logistic Regression Probabilities
# # ==========================================

# # X_test must contain the same test samples
# # in exactly the same order as test_loader.

# lr_probs = clf.predict_proba(X_test)

# print("Logistic Regression probability shape:", lr_probs.shape)


# # ==========================================
# # 3. Soft Voting Ensemble
# # ==========================================

# # Equal weights: 50% DeBERTa + 50% Logistic Regression

# ensemble_probs = (
#     0.5 * deberta_probs +
#     0.5 * lr_probs
# )

# ensemble_preds = np.argmax(ensemble_probs, axis=1)


# # ==========================================
# # 4. Evaluate Ensemble
# # ==========================================

# print("\n========== ENSEMBLE RESULTS ==========")

# print(
#     "Accuracy:",
#     accuracy_score(y_true, ensemble_preds)
# )

# print(
#     "Macro Recall:",
#     recall_score(
#         y_true,
#         ensemble_preds,
#         average="macro"
#     )
# )

# print(
#     "Confusion Matrix:\n",
#     confusion_matrix(y_true, ensemble_preds)
# )




# Weighted Soft Voting
# DeBERTa gets 70% weight
# Logistic Regression gets 30% weight

# w_deberta = 0.7
# w_lr = 0.3


# weights = [
#     (0.5, 0.5),
#     (0.6, 0.4),
#     (0.7, 0.3),
#     (0.8, 0.2),
#     (0.9, 0.1)
# ]

# for w_deberta, w_lr in weights:

#     probs = (
#         w_deberta * deberta_probs +
#         w_lr * lr_probs
#     )

#     preds = np.argmax(probs, axis=1)

#     acc = accuracy_score(y_true, preds)

#     recall = recall_score(
#         y_true,
#         preds,
#         average="macro"
#     )

#     print(
#         f"DeBERTa={w_deberta:.1f}, "
#         f"LR={w_lr:.1f} | "
#         f"Accuracy={acc:.4f} | "
#         f"Macro Recall={recall:.4f}"
#     )